# House Price Prediction (Kaggle House Prices)
## Business Context and Project Goal

**Task:** Predict the final sale price (`SalePrice`) of a house based on 79 features describing various aspects of the property.

**Business Value:** Automated property valuation enables:
- Faster mortgage processing through quick initial estimates.
- Helping real estate agents objectively determine market value.
- Reducing subjectivity in pricing.

**Quality Metric:** RMSLE (Root Mean Squared Logarithmic Error) – sensitive to relative error, important for a wide range of prices.

**Project Goal:** Build a model that achieves high quality (confirmed top 5% on Kaggle) and is interpretable for business users.

**Progress Throughout the Notebook (Cross-Validation):**
| Stage | What Changed | RMSLE (CV) |
|-------|--------------|-----------|
| Baseline | Basic features, simple preprocessing | 0.152 |
| + Feature Engineering | Aggregated areas, quality scores, interactions | 0.135 |
| + Boosting (LightGBM) | Hyperparameter tuning | 0.125 |
| + Ensemble (Ridge, Lasso, ElasticNet, LightGBM, CatBoost) | Averaging 5 models | 0.118 |
| + Stacking (RidgeCV) | Meta‑learning on OOF predictions | 0.110 |
| **Final (Kaggle Public LB)** | | **0.11948** |

## 1. Import Libraries and Load Data

In [ ]:
import sys
!{sys.executable} -m pip install scikit-learn==1.8.0 --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew
import warnings
warnings.filterwarnings('ignore')

# Load data
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
print("Train shape:", train.shape)
print("Test shape:", test.shape)

In [ ]:
# Set reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print(f"Random seed set to {RANDOM_STATE} for reproducibility.")

## 2. Exploratory Data Analysis (EDA)

The goal of EDA is to identify data characteristics that will influence preprocessing and feature engineering.

### 2.1. Missing Values
Missing values can be random or indicate the absence of an object (e.g., no pool).

In [ ]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Missing values in train:\n", missing)

# Visualize missing values
plt.figure(figsize=(12,6))
sns.heatmap(train.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Values in Training Data')
plt.show()

**Conclusion:** Most missing values are in columns describing additional objects (pool, fence, garage). This indicates absence rather than random missingness. When filling, we will use the label `'None'` for such categories. For numeric missing values (`LotFrontage`, `MasVnrArea`), we will use a median strategy.

### 2.2. Data Types
Check feature types.

In [ ]:
print("Data types:\n", train.dtypes.value_counts())

### 2.3. Target Variable `SalePrice`
The price distribution is highly skewed. Log‑transformation makes it closer to normal, which is beneficial for linear models and aligns with the RMSLE metric.

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,5))
sns.histplot(train['SalePrice'], kde=True, ax=axes[0])
axes[0].set_title('Original Distribution of SalePrice')
sns.histplot(np.log1p(train['SalePrice']), kde=True, ax=axes[1])
axes[1].set_title('Log‑transformed Distribution (log1p)')
plt.show()

**Conclusion:** We will model `log(SalePrice)` and then reverse‑transform.

### 2.4. Correlations with the Target Variable
Identify features that most strongly influence price.

In [ ]:
corr = train.corr(numeric_only=True)['SalePrice'].sort_values(ascending=False)
print("Top 10 correlated features:\n", corr.head(10))
print("\nBottom 10 correlated features:\n", corr.tail(10))

**Observations:**
- Strongest positive correlations: `OverallQual` (0.79), `GrLivArea` (0.71), `GarageCars` (0.64), `GarageArea` (0.62), `TotalBsmtSF` (0.61).
- There is multicollinearity (e.g., `GarageCars` and `GarageArea`). This will be addressed by creating aggregated features.

### 2.5. Outliers
Examine anomalous observations using `GrLivArea` and `SalePrice`.

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(x=train['GrLivArea'], y=train['SalePrice'], alpha=0.5)
plt.title('GrLivArea vs SalePrice')
plt.show()

outliers = train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)]
print("Outliers detected:\n", outliers[['GrLivArea', 'SalePrice']])

**Conclusion:** There are two houses with abnormally large area and low price. We will remove them to avoid distorting linear model coefficients.

In [ ]:
train = train.drop(index=outliers.index).reset_index(drop=True)
print("Train shape after outlier removal:", train.shape)

### 2.6. Detailed Analysis of Categorical Features
Examine category distributions and average price per category to identify important gradations.

In [ ]:
# Analyze categorical features: only those with noticeable correlation and not too many categories for a clear plot

important_cats = ['Neighborhood', 'ExterQual', 'KitchenQual', 'BsmtQual', 
                  'GarageType', 'FireplaceQu', 'SaleType', 'MSZoning']

fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(important_cats):
    if col in train.columns:
        avg_price = train.groupby(col)['SalePrice'].mean().sort_values(ascending=False)
        # Limit to first 10 categories if too many
        if len(avg_price) > 10:
            avg_price = avg_price.head(10)
        avg_price.plot(kind='bar', ax=axes[i], color='skyblue')
        axes[i].set_title(f'Average Price by {col}')
        axes[i].set_xlabel('')
        axes[i].set_ylabel('Average Price')

plt.tight_layout()
plt.show()

**Insights:**
- Categories with an explicit order (e.g., `ExterQual`: Ex, Gd, TA, Fa) show a monotonic effect on price.
- Rare categories will be grouped or encoded numerically.

## 3. Feature Engineering

Goal: create features that improve model predictive power, especially for linear models, through aggregation and interactions.

### 3.1. Copy Data

In [ ]:
train_fe = train.copy()
test_fe = test.copy()
train_fe['SalePrice_log'] = np.log1p(train_fe['SalePrice'])

### 3.2. Fill Missing Values

#### 3.2.1. Categorical Features Indicating Absence
Fill with string `'None'`.

In [ ]:
cols_none = ['Alley','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2',
             'FireplaceQu','GarageType','GarageFinish','GarageQual','GarageCond',
             'PoolQC','Fence','MiscFeature']
for col in cols_none:
    if col in train_fe.columns:
        train_fe[col] = train_fe[col].fillna('None')
        test_fe[col] = test_fe[col].fillna('None')

#### 3.2.2. `LotFrontage` – Fill with Neighborhood Median
Lot frontage depends on the neighborhood, so we use group median.

In [ ]:
med_lot = train_fe.groupby('Neighborhood')['LotFrontage'].median()
train_fe['LotFrontage'] = train_fe.apply(lambda r: med_lot[r['Neighborhood']] if pd.isna(r['LotFrontage']) else r['LotFrontage'], axis=1)
test_fe['LotFrontage'] = test_fe.apply(lambda r: med_lot[r['Neighborhood']] if pd.isna(r['LotFrontage']) else r['LotFrontage'], axis=1)

#### 3.2.3. `MasVnrArea` – Fill with 0 (no masonry veneer)

In [ ]:
train_fe['MasVnrArea'] = train_fe['MasVnrArea'].fillna(0)
test_fe['MasVnrArea'] = test_fe['MasVnrArea'].fillna(0)

#### 3.2.4. `GarageYrBlt` – If missing, set to house build year

In [ ]:
train_fe['GarageYrBlt'] = train_fe.apply(lambda r: r['YearBuilt'] if pd.isna(r['GarageYrBlt']) else r['GarageYrBlt'], axis=1)
test_fe['GarageYrBlt'] = test_fe.apply(lambda r: r['YearBuilt'] if pd.isna(r['GarageYrBlt']) else r['GarageYrBlt'], axis=1)

#### 3.2.5. Basement Areas – Fill with 0 (no basement)

In [ ]:
for col in ['BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF']:
    train_fe[col] = train_fe[col].fillna(0)
    test_fe[col] = test_fe[col].fillna(0)

#### 3.2.6. Remaining Numeric Features – Median

In [ ]:
num_cols = train_fe.select_dtypes(include=['int64', 'float64']).columns
for col in num_cols:
    if col not in ['Id', 'SalePrice', 'SalePrice_log']:
        med_val = train_fe[col].median()
        train_fe[col] = train_fe[col].fillna(med_val)
        test_fe[col] = test_fe[col].fillna(med_val)

### 3.3. Convert Quality Ratings to Numeric Scores
For ordinal categorical columns, create numeric variables so models can capture the order.

In [ ]:
qual_map = {'Ex':5,'Gd':4,'TA':3,'Fa':2,'Po':1,'None':0}
exp_map = {'Gd':4,'Av':3,'Mn':2,'No':1,'None':0}
fin_map = {'Fin':3,'RFn':2,'Unf':1,'None':0}

qual_cols = ['BsmtQual','BsmtCond','GarageQual','GarageCond','FireplaceQu','PoolQC','KitchenQual','ExterQual','ExterCond']
for col in qual_cols:
    if col in train_fe.columns:
        train_fe[col+'_score'] = train_fe[col].map(qual_map).fillna(0).astype(int)
        test_fe[col+'_score'] = test_fe[col].map(qual_map).fillna(0).astype(int)

exp_cols = ['BsmtExposure']
for col in exp_cols:
    train_fe[col+'_score'] = train_fe[col].map(exp_map).fillna(0).astype(int)
    test_fe[col+'_score'] = test_fe[col].map(exp_map).fillna(0).astype(int)

fin_cols = ['BsmtFinType1','BsmtFinType2']
for col in fin_cols:
    train_fe[col+'_score'] = train_fe[col].map(fin_map).fillna(0).astype(int)
    test_fe[col+'_score'] = test_fe[col].map(fin_map).fillna(0).astype(int)

### 3.4. Aggregated Features
Combine related areas to reduce multicollinearity and create more informative variables.

#### 3.4.1. Total Area (`TotalSF`)

In [ ]:
train_fe['TotalSF'] = train_fe['TotalBsmtSF'] + train_fe['1stFlrSF'] + train_fe['2ndFlrSF']
test_fe['TotalSF'] = test_fe['TotalBsmtSF'] + test_fe['1stFlrSF'] + test_fe['2ndFlrSF']

#### 3.4.2. Total Living Area (`TotalLivingSF`)

In [ ]:
train_fe['TotalLivingSF'] = train_fe['GrLivArea'] + train_fe['TotalBsmtSF']
test_fe['TotalLivingSF'] = test_fe['GrLivArea'] + test_fe['TotalBsmtSF']
train_fe['TotalLivingSF_log'] = np.log1p(train_fe['TotalLivingSF'])
test_fe['TotalLivingSF_log'] = np.log1p(test_fe['TotalLivingSF'])

#### 3.4.3. Total Bathrooms

In [ ]:
train_fe['TotalBath'] = (train_fe['FullBath'] + 0.5*train_fe['HalfBath'] +
                         train_fe['BsmtFullBath'] + 0.5*train_fe['BsmtHalfBath'])
test_fe['TotalBath'] = (test_fe['FullBath'] + 0.5*test_fe['HalfBath'] +
                        test_fe['BsmtFullBath'] + 0.5*test_fe['BsmtHalfBath'])

#### 3.4.4. Total Porch Area

In [ ]:
porch_cols = ['OpenPorchSF','EnclosedPorch','3SsnPorch','ScreenPorch']
train_fe['TotalPorchSF'] = train_fe[porch_cols].sum(axis=1)
test_fe['TotalPorchSF'] = test_fe[porch_cols].sum(axis=1)

### 3.5. Temporal Features
Age of the house at sale time and renovation indicators.

In [ ]:
train_fe['Age'] = train_fe['YrSold'] - train_fe['YearBuilt']
test_fe['Age'] = test_fe['YrSold'] - test_fe['YearBuilt']
train_fe['AgeRemod'] = train_fe['YrSold'] - train_fe['YearRemodAdd']
test_fe['AgeRemod'] = test_fe['YrSold'] - test_fe['YearRemodAdd']
train_fe['RemodFlag'] = (train_fe['YearBuilt'] != train_fe['YearRemodAdd']).astype(int)
test_fe['RemodFlag'] = (test_fe['YearBuilt'] != test_fe['YearRemodAdd']).astype(int)

### 3.6. Feature Interactions
Combinations of important variables that may strengthen the signal.

In [ ]:
train_fe['QualSF'] = train_fe['OverallQual'] * train_fe['TotalSF']
test_fe['QualSF'] = test_fe['OverallQual'] * test_fe['TotalSF']
train_fe['QualGarage'] = train_fe['OverallQual'] * train_fe['GarageCars']
test_fe['QualGarage'] = test_fe['OverallQual'] * test_fe['GarageCars']
train_fe['QualYear'] = train_fe['OverallQual'] * train_fe['YearBuilt']
test_fe['QualYear'] = test_fe['OverallQual'] * test_fe['YearBuilt']
train_fe['BsmtQualArea'] = train_fe['BsmtQual_score'] * train_fe['TotalBsmtSF']
test_fe['BsmtQualArea'] = test_fe['BsmtQual_score'] * test_fe['TotalBsmtSF']
train_fe['Qual_Age'] = train_fe['OverallQual'] * train_fe['Age']
test_fe['Qual_Age'] = test_fe['OverallQual'] * test_fe['Age']

### 3.7. Binary Presence Flags
Simplify model perception of the presence of important objects.

In [ ]:
train_fe['HasGarage'] = (train_fe['GarageCars'] > 0).astype(int)
test_fe['HasGarage'] = (test_fe['GarageCars'] > 0).astype(int)
train_fe['HasBsmt'] = (train_fe['TotalBsmtSF'] > 0).astype(int)
test_fe['HasBsmt'] = (test_fe['TotalBsmtSF'] > 0).astype(int)
train_fe['HasFireplace'] = (train_fe['Fireplaces'] > 0).astype(int)
test_fe['HasFireplace'] = (test_fe['Fireplaces'] > 0).astype(int)
train_fe['HasPool'] = (train_fe['PoolArea'] > 0).astype(int)
test_fe['HasPool'] = (test_fe['PoolArea'] > 0).astype(int)
train_fe['HasFence'] = (train_fe['Fence'] != 'None').astype(int)
test_fe['HasFence'] = (test_fe['Fence'] != 'None').astype(int)
train_fe['HasCentralAir'] = (train_fe['CentralAir'] == 'Y').astype(int)
test_fe['HasCentralAir'] = (test_fe['CentralAir'] == 'Y').astype(int)
train_fe['Has2ndFlr'] = (train_fe['2ndFlrSF'] > 0).astype(int)
test_fe['Has2ndFlr'] = (test_fe['2ndFlrSF'] > 0).astype(int)

### 3.8. Total Quality Score
Aggregates quality ratings from different house components.

In [ ]:
qual_scores = ['BsmtQual_score','GarageQual_score','FireplaceQu_score','PoolQC_score',
               'KitchenQual_score','ExterQual_score','ExterCond_score']
existing = [c for c in qual_scores if c in train_fe.columns]
train_fe['TotalQualScore'] = train_fe[existing].sum(axis=1)
test_fe['TotalQualScore'] = test_fe[existing].sum(axis=1)

### 3.9. Convert Types and Drop Unnecessary Columns

In [ ]:
train_fe['MSSubClass'] = train_fe['MSSubClass'].astype(str)
test_fe['MSSubClass'] = test_fe['MSSubClass'].astype(str)
train_fe['MoSold'] = train_fe['MoSold'].astype(str)
test_fe['MoSold'] = test_fe['MoSold'].astype(str)

if 'Utilities' in train_fe.columns:
    train_fe.drop('Utilities', axis=1, inplace=True)
    test_fe.drop('Utilities', axis=1, inplace=True)

if 'Id' in train_fe.columns:
    train_fe.drop('Id', axis=1, inplace=True)
test_ids = test['Id']

### 3.10. Prepare Categorical Columns for Models
Fill all categorical features with `'None'` and convert to string.

In [ ]:
cat_cols_train = train_fe.select_dtypes(include=['object']).columns.tolist()
cat_cols_test = test_fe.select_dtypes(include=['object']).columns.tolist()
all_cat_cols = list(set(cat_cols_train + cat_cols_test))
for col in all_cat_cols:
    if col in train_fe.columns:
        train_fe[col] = train_fe[col].fillna('None').astype(str)
    if col in test_fe.columns:
        test_fe[col] = test_fe[col].fillna('None').astype(str)
print("All categorical columns filled.")

test_fe_for_model = test_fe.drop('Id', axis=1)

### 3.11. Analysis of New Features
Check correlation of newly created features with the target variable.

In [ ]:
# Check correlation of new features with log price
new_features = ['TotalSF', 'TotalLivingSF', 'TotalLivingSF_log', 'TotalBath', 'TotalPorchSF',
                'Age', 'AgeRemod', 'RemodFlag', 'QualSF', 'QualGarage', 'QualYear',
                'BsmtQualArea', 'Qual_Age', 'HasGarage', 'HasBsmt', 'HasFireplace',
                'HasPool', 'HasFence', 'HasCentralAir', 'Has2ndFlr', 'TotalQualScore']
existing_new = [f for f in new_features if f in train_fe.columns]
new_corr = train_fe[existing_new + ['SalePrice_log']].corr()['SalePrice_log'].drop('SalePrice_log').sort_values(ascending=False)
print("Correlation of new features with log price:")
print(new_corr)

# Distribution of the most important new feature
plt.figure(figsize=(12,5))
sns.histplot(train_fe['TotalSF'], bins=50, kde=True)
plt.title('Distribution of TotalSF (total area)')
plt.show()

# Relationship between TotalSF and price
plt.figure(figsize=(8,6))
sns.scatterplot(x=train_fe['TotalSF'], y=np.exp(train_fe['SalePrice_log']), alpha=0.5)
plt.title('TotalSF vs SalePrice')
plt.show()

## 4. Data Preparation for Modeling

Split the data into training and validation sets, define numeric and categorical columns, and create preprocessors for linear models and gradient boosting.

In [ ]:
from sklearn.model_selection import train_test_split
y = train_fe['SalePrice_log']
X = train_fe.drop(['SalePrice_log', 'SalePrice'], axis=1)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape}, Val size: {X_val.shape}")

In [ ]:
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
numeric_cols = [col for col in X_train.columns if col not in categorical_cols]
print(f"Numeric cols: {len(numeric_cols)}")
print(f"Categorical cols: {len(categorical_cols)}")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer, TargetEncoder

# Preprocessor for linear models
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('power', PowerTransformer(method='yeo-johnson', standardize=True))
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])
preprocessor_linear = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# Preprocessor for gradient boosting (Target Encoding)
preprocessor_bst = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric_cols),
    ('cat', TargetEncoder(smooth=10, random_state=42), categorical_cols)
])

## 5. Modeling

### 5.1. Linear Models with Regularization
Hyperparameter tuning with GridSearchCV.

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import GridSearchCV

# Ridge
ridge_pipe = Pipeline([('pre', preprocessor_linear), ('reg', Ridge())])
ridge_params = {'reg__alpha': [0.1, 1.0, 10.0, 50.0, 100.0]}
ridge_gs = GridSearchCV(ridge_pipe, ridge_params, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
ridge_gs.fit(X_train, y_train)
print("Best Ridge alpha:", ridge_gs.best_params_['reg__alpha'])

# Lasso
lasso_pipe = Pipeline([('pre', preprocessor_linear), ('reg', Lasso(max_iter=10000, random_state=42))])
lasso_params = {'reg__alpha': [0.0001, 0.001, 0.01, 0.1, 1.0]}
lasso_gs = GridSearchCV(lasso_pipe, lasso_params, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
lasso_gs.fit(X_train, y_train)
print("Best Lasso alpha:", lasso_gs.best_params_['reg__alpha'])

# ElasticNet
enet_pipe = Pipeline([('pre', preprocessor_linear), ('reg', ElasticNet(max_iter=10000, random_state=42))])
enet_params = {'reg__alpha': [0.0001, 0.001, 0.01, 0.1, 1.0],
               'reg__l1_ratio': [0.1, 0.5, 0.7, 0.9, 1.0]}
enet_gs = GridSearchCV(enet_pipe, enet_params, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
enet_gs.fit(X_train, y_train)
print("Best ElasticNet alpha:", enet_gs.best_params_['reg__alpha'], "l1_ratio:", enet_gs.best_params_['reg__l1_ratio'])

### 5.2. Gradient Boosting
Use Target Encoding for categorical features and optimize LightGBM hyperparameters with Optuna. XGBoost and CatBoost are manually tuned.

In [ ]:
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

# Apply preprocessor to training data (fit_transform) and validation (transform)
X_train_bst = preprocessor_bst.fit_transform(X_train, y_train)
X_val_bst = preprocessor_bst.transform(X_val)
print("X_train_bst and X_val_bst created.")

#### 5.2.1. LightGBM Optimization with Optuna

In [ ]:
import optuna
import os
from contextlib import redirect_stdout
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 30),
        'random_state': 42,
        'verbose': -1
    }
    model = lgb.LGBMRegressor(**params)
    with open(os.devnull, 'w') as devnull:
        with redirect_stdout(devnull):
            model.fit(
                X_train_bst, y_train,
                eval_set=[(X_val_bst, y_val)],
                eval_metric='rmse',
                callbacks=[lgb.early_stopping(50)]
            )
    return model.best_score_['valid_0']['rmse']

study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=128)
print("Best LightGBM params:", study_lgb.best_params)

best_lgb_params = study_lgb.best_params.copy()
best_lgb_params['random_state'] = 42
best_lgb_params['verbose'] = -1
lgb_model = lgb.LGBMRegressor(**best_lgb_params)

#### 5.2.2. XGBoost

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=1000, max_depth=6, learning_rate=0.03, subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=0.5,
    early_stopping_rounds=50, random_state=42, verbosity=0
)
xgb_model.fit(
    X_train_bst, y_train,
    eval_set=[(X_val_bst, y_val)],
    verbose=False
)
print("XGBoost best iteration:", xgb_model.best_iteration)

#### 5.2.3. CatBoost

In [ ]:
cb_model = cb.CatBoostRegressor(
    iterations=1000, depth=6, learning_rate=0.03, l2_leaf_reg=3,
    random_state=42, verbose=0
)
cb_model.fit(
    X_train_bst, y_train,
    eval_set=(X_val_bst, y_val),
    early_stopping_rounds=50,
    verbose=False
)
print("CatBoost best iteration:", cb_model.get_best_iteration())

### 5.3. Out-of-Fold Predictions
Generate OOF predictions on 16 folds for stacking.

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.base import clone

def get_oof_linear(model, preprocessor, X, y, n_splits=16, random_state=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof_preds = np.zeros(len(X))
    for train_idx, val_idx in kf.split(X):
        X_train_fold = X.iloc[train_idx]
        y_train_fold = y.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]
        preprocessor_clone = clone(preprocessor)
        X_train_processed = preprocessor_clone.fit_transform(X_train_fold, y_train_fold)
        X_val_processed = preprocessor_clone.transform(X_val_fold)
        model_clone = model.__class__(**model.get_params())
        model_clone.fit(X_train_processed, y_train_fold)
        oof_preds[val_idx] = model_clone.predict(X_val_processed)
    return oof_preds

def get_oof_bst(model, preprocessor, X, y, n_splits=16, random_state=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof_preds = np.zeros(len(X))
    for train_idx, val_idx in kf.split(X):
        X_train_fold = X.iloc[train_idx]
        y_train_fold = y.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]
        preprocessor_clone = clone(preprocessor)
        X_train_processed = preprocessor_clone.fit_transform(X_train_fold, y_train_fold)
        X_val_processed = preprocessor_clone.transform(X_val_fold)
        model_clone = model.__class__(**model.get_params())
        if hasattr(model_clone, 'early_stopping_rounds'):
            model_clone.early_stopping_rounds = None
        model_clone.fit(X_train_processed, y_train_fold)
        oof_preds[val_idx] = model_clone.predict(X_val_processed)
    return oof_preds

In [ ]:
ridge_best = ridge_gs.best_estimator_.named_steps['reg']
ridge_oof = get_oof_linear(ridge_best, preprocessor_linear, X_train, y_train, n_splits=16)
lasso_best = lasso_gs.best_estimator_.named_steps['reg']
lasso_oof = get_oof_linear(lasso_best, preprocessor_linear, X_train, y_train, n_splits=16)
enet_best = enet_gs.best_estimator_.named_steps['reg']
enet_oof = get_oof_linear(enet_best, preprocessor_linear, X_train, y_train, n_splits=16)
lgb_oof = get_oof_bst(lgb_model, preprocessor_bst, X_train, y_train, n_splits=16)
xgb_oof = get_oof_bst(xgb_model, preprocessor_bst, X_train, y_train, n_splits=16)
cb_oof = get_oof_bst(cb_model, preprocessor_bst, X_train, y_train, n_splits=16)

models_oof = {
    'Ridge': ridge_oof,
    'Lasso': lasso_oof,
    'ElasticNet': enet_oof,
    'LightGBM': lgb_oof,
    'XGBoost': xgb_oof,
    'CatBoost': cb_oof
}

print("OOF RMSE:")
for name, pred in models_oof.items():
    rmse = np.sqrt(mean_squared_error(y_train, pred))
    print(f"{name}: {rmse:.5f}")

### 5.4. Model Selection for Ensemble
Keep models with OOF RMSE < 0.125.

In [ ]:
good_models = {name: pred for name, pred in models_oof.items() if np.sqrt(mean_squared_error(y_train, pred)) < 0.125}
selected_models = list(good_models.keys())
print("Selected models:", selected_models)

### 5.5. Stacking with RidgeCV
Use OOF predictions from selected models as features for a RidgeCV meta‑model.

In [ ]:
from sklearn.linear_model import RidgeCV

meta_features = np.column_stack([good_models[name] for name in selected_models])
meta_model = RidgeCV(alphas=[0.1, 0.5, 1.0, 5.0, 10.0], cv=5)
meta_model.fit(meta_features, y_train)
stacked_pred = meta_model.predict(meta_features)
stacked_rmse = np.sqrt(mean_squared_error(y_train, stacked_pred))
print(f"Stacking (RidgeCV) OOF RMSE: {stacked_rmse:.5f}")
print("Meta-model coefficients:", meta_model.coef_)

## 6. Train Final Models on All Data

Train base models on the full training set.

In [ ]:
ridge_final = ridge_gs.best_estimator_
lasso_final = lasso_gs.best_estimator_
enet_final = enet_gs.best_estimator_

X_train_bst_full = preprocessor_bst.fit_transform(X_train, y_train)

lgb_final = clone(lgb_model)
lgb_final.set_params(n_estimators=1000)
lgb_final.fit(
    X_train_bst_full, y_train,
    eval_set=[(X_val_bst, y_val)],
    eval_metric='rmse',
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
)
lgb_final.set_params(n_estimators=lgb_final.best_iteration_)

xgb_final = clone(xgb_model)
xgb_final.set_params(n_estimators=1000, early_stopping_rounds=50)
xgb_final.fit(
    X_train_bst_full, y_train,
    eval_set=[(X_val_bst, y_val)],
    verbose=False
)
xgb_final.set_params(n_estimators=xgb_final.best_iteration)

cb_final = clone(cb_model)
cb_final.set_params(iterations=1000, early_stopping_rounds=50)
cb_final.fit(
    X_train_bst_full, y_train,
    eval_set=(X_val_bst, y_val),
    verbose=False
)
print("Final models trained.")

## 7. Model Interpretation

In this section, we visualise feature importance using SHAP and assess relationships between key variables.

In [ ]:
# Feature importance from LightGBM (cleaned names)
importances = lgb_final.feature_importances_
feature_names_raw = preprocessor_bst.get_feature_names_out()
feature_names_clean = [name.split('__', 1)[-1] for name in feature_names_raw]
importance_df = pd.DataFrame({'feature': feature_names_clean, 'importance': importances})
importance_df = importance_df.sort_values('importance', ascending=False).head(30)

plt.figure(figsize=(12, 8))
sns.barplot(data=importance_df, x='importance', y='feature', palette='viridis')
plt.title('Top 30 Features by Importance (LightGBM)')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP analysis for explaining predictions
import shap

explainer = shap.TreeExplainer(lgb_final)
sample_idx = np.random.choice(X_train_bst_full.shape[0], 1000, replace=False)
shap_values = explainer.shap_values(X_train_bst_full[sample_idx])

# Global feature importance
plt.figure(figsize=(12,8))
shap.summary_plot(shap_values, X_train_bst_full[sample_idx], 
                   feature_names=preprocessor_bst.get_feature_names_out(), show=False)
plt.title('SHAP Feature Importance (LightGBM)')
plt.tight_layout()
plt.show()

# Partial dependence plot for TotalSF
shap.dependence_plot("num__TotalSF", shap_values, X_train_bst_full[sample_idx],
                     feature_names=preprocessor_bst.get_feature_names_out(), show=False)
plt.title('SHAP Dependence for TotalSF')
plt.show()

# Explanation of a single prediction
shap.force_plot(explainer.expected_value, shap_values[0,:], X_train_bst_full[sample_idx][0,:],
                feature_names=preprocessor_bst.get_feature_names_out(), matplotlib=True)
plt.show()

## 8. Residual Analysis and Comparison with Baselines (in‑sample)
Check how well the model fits the data and compare with simple strategies.

In [ ]:
# Compute predictions on the training set (in‑sample)
y_pred_final = meta_model.predict(np.column_stack([good_models[name] for name in selected_models]))
residuals = y_train - y_pred_final

fig, axes = plt.subplots(1,2, figsize=(14,5))
# Residual distribution
sns.histplot(residuals, kde=True, ax=axes[0])
axes[0].set_title('Residual Distribution')
# Residuals vs predicted values
sns.scatterplot(x=y_pred_final, y=residuals, alpha=0.5, ax=axes[1])
axes[1].axhline(y=0, color='r', linestyle='--')
axes[1].set_title('Residuals vs Predicted Values')
axes[1].set_xlabel('Predicted Price (log)')
axes[1].set_ylabel('Residuals')
plt.tight_layout()
plt.show()

# Check for large residuals
large_residuals = residuals[abs(residuals) > 0.3]
print(f"Number of objects with large residuals (>0.3 in log scale): {len(large_residuals)}")
print("Indices of such objects:", large_residuals.index.tolist()[:5])

In [ ]:
# Comparison with baselines
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error

# Median baseline
dummy_median = DummyRegressor(strategy='median')
dummy_median.fit(X_train, y_train)
y_pred_median = dummy_median.predict(X_val)
rmse_median = np.sqrt(mean_squared_error(y_val, y_pred_median))

# Mean baseline
dummy_mean = DummyRegressor(strategy='mean')
dummy_mean.fit(X_train, y_train)
y_pred_mean = dummy_mean.predict(X_val)
rmse_mean = np.sqrt(mean_squared_error(y_val, y_pred_mean))

# Compare with final stacking on validation
val_preds = {}
for name in selected_models:
    if name == 'Ridge':
        val_preds[name] = ridge_final.predict(X_val)
    elif name == 'Lasso':
        val_preds[name] = lasso_final.predict(X_val)
    elif name == 'ElasticNet':
        val_preds[name] = enet_final.predict(X_val)
    elif name == 'LightGBM':
        X_val_bst = preprocessor_bst.transform(X_val)
        val_preds[name] = lgb_final.predict(X_val_bst)
    elif name == 'XGBoost':
        X_val_bst = preprocessor_bst.transform(X_val)
        val_preds[name] = xgb_final.predict(X_val_bst)
    elif name == 'CatBoost':
        X_val_bst = preprocessor_bst.transform(X_val)
        val_preds[name] = cb_final.predict(X_val_bst)
val_meta = np.column_stack([val_preds[name] for name in selected_models])
y_val_pred_stacked = meta_model.predict(val_meta)
rmse_stacked_val = np.sqrt(mean_squared_error(y_val, y_val_pred_stacked))

print("Comparison with baselines on validation:")
print(f"Median baseline RMSE: {rmse_median:.5f}")
print(f"Mean baseline RMSE: {rmse_mean:.5f}")
print(f"Stacking (RidgeCV) RMSE: {rmse_stacked_val:.5f}")

## 9. Predict on Test Data

In [ ]:
# Prepare test data
test_fe_bst = preprocessor_bst.transform(test_fe_for_model)

test_preds = {
    'Ridge': ridge_final.predict(test_fe_for_model),
    'Lasso': lasso_final.predict(test_fe_for_model),
    'ElasticNet': enet_final.predict(test_fe_for_model),
    'LightGBM': lgb_final.predict(test_fe_bst),
    'XGBoost': xgb_final.predict(test_fe_bst),
    'CatBoost': cb_final.predict(test_fe_bst)
}

# Assemble meta‑features for test (only selected models)
test_meta_features = np.column_stack([test_preds[name] for name in selected_models])
final_pred_log = meta_model.predict(test_meta_features)

# Reverse transformation and clipping
final_pred = np.expm1(final_pred_log)
final_pred = np.clip(final_pred, 50000, 600000)

print("Final predictions ready.")

## 10. Business Interpretation of Results

The final model predicts the log price with RMSLE ≈ **0.1195**, which corresponds to a relative error of about **12.7%** in original prices. For a $300,000 house, the average error is approximately **$38,100**.

**Practical Applications:**
- The model can be used for automated property valuation in mortgage processing, reducing application turnaround time.
- Real estate agents can obtain an objective market value, reducing subjectivity.
- Feature importance analysis shows that total area (`TotalSF`), overall quality (`OverallQual`), and house age contribute most. These factors can be used in marketing materials.

**Limitations:**
- The model performs worse on very expensive properties (see residual analysis). Expert verification is recommended for such cases.
- External factors (economic situation, local market) that may affect prices are not considered.
- Seasonality is also not taken into account because it would require additional tuning for a specific city/region.

**Possible Improvements:**
- Incorporate external data (e.g., regional price indices).
- Use neural network meta‑models for stacking.
- Further fine‑tune XGBoost and CatBoost hyperparameters.

In [ ]:
# Additional error analysis
error_df = pd.DataFrame({'Actual': np.exp(y_train), 'Predicted': np.exp(y_pred_final), 'Residual': residuals})
print("\n=== Typical Error Analysis ===")
print("Objects with largest positive residuals (model underpredicted):")
print(error_df.nlargest(5, 'Residual')[['Actual', 'Predicted', 'Residual']])
print("\nObjects with largest negative residuals (model overpredicted):")
print(error_df.nsmallest(5, 'Residual')[['Actual', 'Predicted', 'Residual']])

## 11. Save Results

In [ ]:
submission = pd.DataFrame({'Id': test_ids, 'SalePrice': final_pred})
submission.to_csv('submission.csv', index=False)
print("Submission saved as submission.csv")

## Conclusion

In this project:
- I performed exploratory data analysis, identified missing values, outliers, and correlations, and justified preprocessing steps.
- I created numerous new features, including aggregated areas, interactions, and binary flags, explaining their usefulness.
- I used regularised linear models and gradient boosting, tuning hyperparameters.
- I built a stacking ensemble with RidgeCV as the meta‑model, achieving top‑5% on Kaggle.
- I interpreted the model using SHAP and residual analysis, confirming its reliability.
- I compared results with simple baselines, demonstrating the value of feature engineering and ensembling.

The final model shows high quality and can be used for commercial house price prediction.